# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-ai98/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
# ML-04 — Search Intelligence Data Contract
# Setup: authenticate to Hugging Face and connect DuckDB

from google.colab import userdata
import duckdb
import pandas as pd

# Get Hugging Face token securely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)

""")

print("Hugging Face authentication configured.")
print("DuckDB connection ready.")

Hugging Face authentication configured.
DuckDB connection ready.


In [26]:
DECISION_MONTH = "2026-03"

START_DATE = "2026-03-01"
END_DATE = "2026-03-31"

FACT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("Decision month:", DECISION_MONTH)
print("Window:", START_DATE, "to", END_DATE)

Decision month: 2026-03
Window: 2026-03-01 to 2026-03-31


## 1. Unit of analysis + time window

**Unit of analysis:** one row = one content page for the March 2026 decision window.

The source warehouse table contains daily content-performance records. For this lane, I will aggregate those daily records to one page-level row for the March 2026 decision window.

**Time window:** `2026-03-01` through `2026-03-31`.

The purpose is to rank content pages by observed evidence that they should be reviewed for actions such as refresh, expansion, protection, pruning, or monitoring.

The starter ranking proxy is a decline signal based on observed search performance. Because the current warehouse table does not directly contain `trend_direction`, I will construct the starter proxy from observed March performance rather than pretending that a future outcome is available.

I will develop on the mid-panel month `2026-03`. The `_sample` table is not used for label development because it represents the final month and should remain a sealed test/outcome period.

In [27]:
print("Unit: one content page")
print("Decision window:", START_DATE, "to", END_DATE)

Unit: one content page
Decision window: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

### Features

I will use five page-level features:

1. `gsc_impressions` — total observed Google Search Console impressions during the decision window.
2. `gsc_clicks` — total observed Google Search Console clicks during the decision window.
3. `gsc_avg_position` — weighted average observed search position during the decision window.
4. `ga4_sessions` — total observed GA4 sessions when GA4 data is available.
5. `ga4_engaged_sessions` — total observed engaged GA4 sessions when GA4 data is available.

### Label / proxy

The starter proxy is `is_declining_label`.

Because the warehouse does not contain a ready-made `trend_direction` field, I will define the proxy from the observed March search signal. A page is considered declining when its later-period search impressions are lower than its earlier-period search impressions within the March window.

This is only a starter directional proxy. It is not a true future outcome.

### Context

- `content_hash_id` — identifies the content page.
- `client_hash_id` — identifies the client.
- `report_date` — identifies the daily observation.
- `month` — identifies the warehouse month.
- `gsc_data_available` — indicates whether GSC data is available.
- `ga4_data_available` — indicates whether GA4 data is available.

### Excluded

I will exclude `is_declining_label` from the honest feature set because it is the label/proxy.

I will also exclude `report_date` and `month` from the model features because they identify the observation window rather than describing the content opportunity itself.

Client and content identifiers are retained as context for grouping and validation, not as predictive features.

Future-period measurements are excluded because they would not be known at the decision moment.

In [28]:
FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

LABEL = "is_declining_label"

CONTEXT_FIELDS = [
    "content_hash_id",
    "client_hash_id",
    "report_date",
    "month",
    "gsc_data_available",
    "ga4_data_available",
]

print("Features:")
for feature in FEATURES:
    print("-", feature)

print("\nLabel/proxy:", LABEL)

print("\nContext:")
for field in CONTEXT_FIELDS:
    print("-", field)

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

Label/proxy: is_declining_label

Context:
- content_hash_id
- client_hash_id
- report_date
- month
- gsc_data_available
- ga4_data_available


## 3. Verify it with queries (grain, counts, missing values, windows)

### Query 1 — Grain

The source table is daily, so multiple rows for the same content page are expected in the raw table. The page-level contract is verified by aggregating the March records by `content_hash_id` and confirming that the resulting page-level frame contains one row per content page.


### Query 2 — Row count and date span

This query checks how many daily warehouse rows are present in the March 2026 development window and verifies the observed minimum and maximum dates.


### Query 3 — Data availability

GA4 measurements are only considered available when `ga4_data_available IS TRUE`.

I will not treat a missing or zero GA4 measurement as evidence that analytics data exists. The availability flag is checked explicitly.

In [29]:
# VERIFICATION QUERY 1 — GRAIN

grain_check = con.sql(f"""
WITH page_level AS (
    SELECT
        content_hash_id
    FROM read_parquet('{FACT_PATH}')
    WHERE report_date >= DATE '{START_DATE}'
      AND report_date <= DATE '{END_DATE}'
    GROUP BY content_hash_id
)
SELECT
    COUNT(*) AS page_rows,
    COUNT(DISTINCT content_hash_id) AS distinct_content_pages
FROM page_level
""").df()

display(grain_check)

assert (
    grain_check.loc[0, "page_rows"]
    == grain_check.loc[0, "distinct_content_pages"]
)

print("Grain check passed: one row per content page.")


# VERIFICATION QUERY 2 — ROW COUNT + DATE SPAN

window_check = con.sql(f"""
SELECT
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT content_hash_id) AS distinct_content_pages,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{FACT_PATH}')
WHERE report_date >= DATE '{START_DATE}'
  AND report_date <= DATE '{END_DATE}'
""").df()

display(window_check)

assert str(window_check.loc[0, "min_report_date"])[:10] == START_DATE
assert str(window_check.loc[0, "max_report_date"])[:10] == END_DATE

print("Window check passed.")


# VERIFICATION QUERY 3 — AVAILABILITY
# Required explicit IS TRUE check.

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NOT TRUE
    ) AS ga4_not_available_rows

FROM read_parquet('{FACT_PATH}')
WHERE report_date >= DATE '{START_DATE}'
  AND report_date <= DATE '{END_DATE}'
""").df()

display(availability_check)

print("Availability check completed using IS TRUE.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,page_rows,distinct_content_pages
0,331437,331437


Grain check passed: one row per content page.


,daily_rows,distinct_content_pages,min_report_date,max_report_date
0,9841378,331437,2026-03-01,2026-03-31


Window check passed.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows,ga4_available_rows,ga4_not_available_rows
0,9841378,3611061,413966,9427412


Availability check completed using IS TRUE.


## Five-feature frame

The starter feature frame contains five page-level features aggregated from the March 2026 daily warehouse data.

- **`gsc_impressions`** — available at the decision moment because the impressions have already been observed in Search Console.
- **`gsc_clicks`** — available at the decision moment because the clicks have already been observed in Search Console.
- **`gsc_avg_position`** — available at the decision moment because it is calculated from observed search-position measurements.
- **`ga4_sessions`** — available at the decision moment when `ga4_data_available IS TRUE`, because the sessions have already been observed.
- **`ga4_engaged_sessions`** — available at the decision moment when `ga4_data_available IS TRUE`, because engaged sessions have already been observed.

The feature frame is aggregated to one row per content page. Identifiers and availability flags are retained separately as context rather than model features.

In [30]:
# Build the five-feature page-level frame

feature_frame = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0
        END
    ) AS gsc_impressions,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS gsc_clicks,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) > 0
        THEN
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )
            /
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )
        ELSE NULL
    END AS gsc_avg_position,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN COALESCE(ga4_sessions, 0)
            ELSE 0
        END
    ) AS ga4_sessions,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN COALESCE(ga4_engaged_sessions, 0)
            ELSE 0
        END
    ) AS ga4_engaged_sessions

FROM read_parquet('{FACT_PATH}')

WHERE report_date >= DATE '{START_DATE}'
  AND report_date <= DATE '{END_DATE}'

GROUP BY
    content_hash_id,
    client_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)

display(feature_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 7)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.298246,0.0,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,6.893301,1.0,0.0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,5.637584,4.0,0.0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,3.214128,0.0,0.0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.535346,3.0,0.0
5,content_05434271b257bb68,client_73cda7b4e4f265ea,1421.0,6.0,6.906404,9.0,0.0
6,content_22610b0934f8825e,client_73cda7b4e4f265ea,67.0,0.0,12.000000,0.0,0.0
7,content_712c365258cee05c,client_73cda7b4e4f265ea,6048.0,23.0,4.931878,8.0,0.0
8,content_5d412fba6e1a2582,client_73cda7b4e4f265ea,223.0,1.0,10.538117,2.0,0.0
9,content_1f380a642aed423b,client_73cda7b4e4f265ea,96.0,1.0,5.864583,8.0,0.0


## Starter declining proxy

The warehouse does not provide `trend_direction`, so I construct a simple observed directional proxy within March.

I compare the first half of March with the second half of March using GSC impressions.

`is_declining_label = 1` when second-half impressions are lower than first-half impressions.

This is deliberately described as a proxy rather than a future prediction target.

In [31]:
# Create the observed directional proxy.
# First half: March 1-15
# Second half: March 16-31

label_frame = con.sql(f"""
WITH daily AS (
    SELECT
        content_hash_id,
        client_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS first_half_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS second_half_impressions

    FROM read_parquet('{FACT_PATH}')

    GROUP BY
        content_hash_id,
        client_hash_id
)

SELECT
    content_hash_id,
    client_hash_id,
    first_half_impressions,
    second_half_impressions,

    CASE
        WHEN second_half_impressions < first_half_impressions
        THEN 1
        ELSE 0
    END AS is_declining_label

FROM daily
""").df()

feature_frame = feature_frame.merge(
    label_frame,
    on=["content_hash_id", "client_hash_id"],
    how="left"
)

print("Feature + proxy frame shape:", feature_frame.shape)

display(
    feature_frame[
        [
            "content_hash_id",
            "client_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "ga4_engaged_sessions",
            "is_declining_label",
        ]
    ].head(10)
)

print("\nProxy distribution:")
print(feature_frame["is_declining_label"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature + proxy frame shape: (331437, 10)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,is_declining_label
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.298246,0.0,0.0,0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,6.893301,1.0,0.0,1
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,5.637584,4.0,0.0,1
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,3.214128,0.0,0.0,1
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.535346,3.0,0.0,1
5,content_05434271b257bb68,client_73cda7b4e4f265ea,1421.0,6.0,6.906404,9.0,0.0,0
6,content_22610b0934f8825e,client_73cda7b4e4f265ea,67.0,0.0,12.000000,0.0,0.0,1
7,content_712c365258cee05c,client_73cda7b4e4f265ea,6048.0,23.0,4.931878,8.0,0.0,0
8,content_5d412fba6e1a2582,client_73cda7b4e4f265ea,223.0,1.0,10.538117,2.0,0.0,1
9,content_1f380a642aed423b,client_73cda7b4e4f265ea,96.0,1.0,5.864583,8.0,0.0,0



Proxy distribution:
is_declining_label
0    264851
1     66586
Name: count, dtype: int64


## Deliberate leakage experiment

I will deliberately include `is_declining_label` in the feature set.

This is label leakage because the feature is the answer itself. A model given this column can obtain an artificially strong score because it does not need to learn the relationship between the observed signals and the target.

After demonstrating the trap, I remove the label and keep the five honest features.

The honest feature set is the one that should be used for future modeling.

In [32]:
# Deliberate leakage

leaky_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "is_declining_label",   # INTENTIONAL LEAK
]

print("Leaky features:")
for feature in leaky_features:
    print("-", feature)

print("\nWARNING: is_declining_label is intentionally included for the leakage demonstration.")

Leaky features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions
- is_declining_label



In [33]:
# Remove the leak

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

assert "is_declining_label" not in honest_features

print("Honest features:")
for feature in honest_features:
    print("-", feature)

print("\nLeak removed successfully.")

Honest features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

Leak removed successfully.


## Leakage score comparison

For the deliberate experiment, I use a simple train/test classifier only to demonstrate the effect of leakage.

The leaky feature set contains the target itself, so its score should be artificially perfect or near-perfect.

The honest feature set removes the label-derived column. Its score is the honest baseline and should be retained for future modeling work.

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model_df = feature_frame[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "is_declining_label",
    ]
].copy()

X = model_df.drop(columns=["is_declining_label"])
y = model_df["is_declining_label"]

X = SimpleImputer(strategy="median").fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

honest_score = accuracy_score(
    y_test,
    model.predict(X_test)
)

print("Honest quick score:", round(honest_score, 4))

Honest quick score: 0.7841


In [35]:
# Intentional leakage experiment

leaky_df = feature_frame[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "is_declining_label",
    ]
].copy()

X_leaky = leaky_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "is_declining_label",
    ]
]

y_leaky = leaky_df["is_declining_label"]

X_leaky = SimpleImputer(strategy="median").fit_transform(X_leaky)

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.25,
    random_state=42,
    stratify=y_leaky
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_score = accuracy_score(
    y_test,
    leaky_model.predict(X_test)
)

print("Leaky quick score:", round(leaky_score, 4))
print("Honest quick score:", round(honest_score, 4))

Leaky quick score: 1.0
Honest quick score: 0.7841


## 4. Data limits

This slice has several important limitations.

First, the warehouse is an unbalanced panel, so different clients can have different amounts of historical data.

Second, GSC and GA4 availability are not uniform. A missing analytics measurement must not automatically be interpreted as zero performance; availability is checked using the explicit availability flags.

Third, the daily performance data contains observations across time, so overlapping windows can create leakage if information from a future outcome period is included in the features.

Fourth, `is_declining_label` is a directional proxy constructed from observed March performance. It is not a true future outcome and cannot establish whether refreshing a page will actually improve performance.

Finally, this notebook develops on March 2026. The final month, June 2026, should remain a sealed test/outcome month and should not be used to develop the label logic.

In [36]:
# Final self-check

assert DECISION_MONTH == "2026-03"
assert len(honest_features) == 5
assert "is_declining_label" not in honest_features

assert feature_frame["content_hash_id"].notna().all()
assert feature_frame["client_hash_id"].notna().all()

print("===================================")
print("ML-04 SELF-CHECK")
print("===================================")
print("Lane: Refresh / Content Opportunity Scoring")
print("Task type: Ranking")
print("Decision month:", DECISION_MONTH)
print("Honest feature count:", len(honest_features))
print("Label excluded from honest features: YES")
print("Page-level feature frame created: YES")
print("Leakage experiment performed: YES")
print("===================================")
print("Self-check passed.")

ML-04 SELF-CHECK
Lane: Refresh / Content Opportunity Scoring
Task type: Ranking
Decision month: 2026-03
Honest feature count: 5
Label excluded from honest features: YES
Page-level feature frame created: YES
Leakage experiment performed: YES
Self-check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.